# 4. óra · Klasszifikáció: a nyers adattól a hangolt modellig
**Adat:** IBM Telco Customer Churn (7 043 ügyfél, ~26,5% lemond). **Cél:** az ügyfeleket churn-kockázat szerint sorba rendezni.

Ez a notebook a slide-ok számait és ábráit generálja (`assets/`). Minden szám innen jön, rögzített seedekkel.
Futtatás a lecke mappájából: `uv run jupyter nbconvert --to notebook --execute demo.ipynb --inplace`.

In [1]:
import json, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from sklearn.model_selection import (train_test_split, StratifiedKFold, RepeatedStratifiedKFold,
                                     cross_val_predict, cross_val_score, cross_validate, RandomizedSearchCV)
from sklearn.compose import ColumnTransformer, make_column_selector as selector
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
                             average_precision_score, roc_auc_score, confusion_matrix,
                             roc_curve, precision_recall_curve)
from xgboost import XGBClassifier
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import make_pipeline as imb_pipeline
from scipy.stats import randint, uniform, loguniform

# A deck színeihez igazítva (assets/deck/deck.css), mint a 3. órán: sötét háttér, validált sötét módú lépések.
SURFACE, INK, INK2, GRID = "#0d2a20", "#e9e4d2", "#b9c2b3", "#30483c"
BLUE, ORANGE = "#3987e5", "#d95926"          # marad / lemond
plt.rcParams.update({"figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2, "axes.edgecolor": GRID,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID, "grid.alpha": .7,
    "font.size": 15, "axes.titlesize": 17, "axes.labelsize": 15, "legend.fontsize": 14, "figure.dpi": 110})
DATA, OUT = Path("data/telco.csv"), Path("assets"); OUT.mkdir(exist_ok=True)
hu = lambda v, nd=2: f"{v:.{nd}f}".replace(".", ",")                      # magyar tizedesvessző
def hu_axes(ax, x=True, y=True):
    f = FuncFormatter(lambda v, _: f"{v:g}".replace(".", ","))
    if x: ax.xaxis.set_major_formatter(f)
    if y: ax.yaxis.set_major_formatter(f)
def keep(fig, name): fig.savefig(OUT / f"{name}.png", dpi=170, bbox_inches="tight", facecolor=SURFACE); plt.close(fig)
NUMS = {}                                                                  # a slide-ok számai

## 1. Nyers adat és EDA

In [2]:
raw = pd.read_csv(DATA)
print(raw.shape); display(raw.head(3)); raw.info()
y_all = (raw.Churn == "Yes").astype(int)
NUMS["n_rows"], NUMS["n_cols"], NUMS["n_churn"], NUMS["churn_rate"] = len(raw), raw.shape[1], int(y_all.sum()), float(y_all.mean())
print(f"churn arány: {y_all.mean():.1%}  -> a 'senki nem mond le' modell {1-y_all.mean():.1%}-ban igaza van")

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [3]:
def rate_bars(col, ax, title):
    r = y_all.groupby(raw[col]).mean().sort_values()
    ax.barh(r.index, r.values, color=BLUE, height=.55)
    for i, v in enumerate(r.values): ax.text(v + .012, i, f"{v:.0%}", va="center", fontsize=15, color=INK)
    ax.axvline(y_all.mean(), color=INK2, ls="--", lw=1.2); ax.set_title(title, loc="left"); ax.set_xlim(0, .62); ax.grid(False); ax.set_xticks([])
    return r
fig, ax = plt.subplots(figsize=(8, 3.6)); r = rate_bars("Contract", ax, f"Churn-arány szerződés szerint (szaggatott: átlag, {y_all.mean():.0%})"); keep(fig, "eda_szerzodes")
NUMS["churn_by_contract"] = r.round(4).to_dict()
fig, axes = plt.subplots(1, 2, figsize=(13, 4)); rate_bars("InternetService", axes[0], "Internetszolgáltatás szerint"); rate_bars("PaymentMethod", axes[1], "Fizetési mód szerint"); plt.tight_layout(); keep(fig, "eda_internet_fizetes")
fig, ax = plt.subplots(figsize=(8.5, 4)); bins = np.arange(0, 76, 3)
ax.hist(raw.tenure[y_all == 0], bins, alpha=.8, color=BLUE, label="marad"); ax.hist(raw.tenure[y_all == 1], bins, alpha=.8, color=ORANGE, label="lemond")
ax.set_xlabel("tenure (hónap)"); ax.set_ylabel("ügyfél"); ax.legend(frameon=False); keep(fig, "eda_tenure")

## 2. Adattisztítás

In [4]:
num_tc = pd.to_numeric(raw.TotalCharges, errors="coerce")
bad = raw[num_tc.isna()]
print(len(bad), "sor nem konvertálható; TotalCharges értékeik:", bad.TotalCharges.unique())
display(bad[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]].head())
print("ezek tenure-je:", bad.tenure.unique(), "| lemondott:", bad.Churn.unique())
print("duplikált customerID:", raw.customerID.duplicated().sum(), "| valódi NaN:", raw.isna().sum().sum())
NUMS["blank_total_charges"] = int(len(bad)); NUMS["blank_tenure_values"] = [int(v) for v in bad.tenure.unique()]; NUMS["blank_churned"] = int((bad.Churn == "Yes").sum())

11 sor nem konvertálható; TotalCharges értékeik: <StringArray>
[' ']
Length: 1, dtype: str


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No


ezek tenure-je: [0] | lemondott: <StringArray>
['No']
Length: 1, dtype: str
duplikált customerID: 0 | valódi NaN: 0


In [5]:
df = raw.copy()
df["TotalCharges"] = pd.to_numeric(df.TotalCharges, errors="coerce").fillna(0.0)   # tenure==0 -> még nem számlázták
df["SeniorCitizen"] = df.SeniorCitizen.map({0: "No", 1: "Yes"})                    # 0/1 jelző, nem szám
y = (df.Churn == "Yes").astype(int)
X = df.drop(columns=["customerID", "Churn"])                                       # az azonosító nem hordoz jelet
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, stratify=y, random_state=42)
# ---- X_test / y_test ZÁROLVA a 7. fejezetig. Addig minden keresztvalidáció a train halmazon. ----
NUMS.update(n_train=int(len(X_train)), n_test=int(len(X_test)), churn_train=float(y_train.mean()), churn_test=float(y_test.mean()))
print(len(X_train), len(X_test), f"| churn train {y_train.mean():.3f} test {y_test.mean():.3f}")

# pandas 3: a szöveges oszlop dtype-ja "str", nem "object" -> szám / nem szám alapján választunk
def make_prep(scale=False):
    return ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), selector(dtype_exclude="number")),
                              ("num", StandardScaler() if scale else "passthrough", selector(dtype_include="number"))])
cv = StratifiedKFold(5, shuffle=True, random_state=0)

5634 1409 | churn train 0.265 test 0.265


## 3. Hogyan értékeljünk egy klasszifikátort?

In [6]:
dummy = make_pipeline(make_prep(), DummyClassifier(strategy="most_frequent"))
pred = cross_val_predict(dummy, X_train, y_train, cv=cv)
NUMS["dummy"] = dict(acc=accuracy_score(y_train, pred), bal=balanced_accuracy_score(y_train, pred), recall=recall_score(y_train, pred))
print("'senki nem mond le':", {k: round(v, 3) for k, v in NUMS["dummy"].items()})
rng = np.random.default_rng(0); y_rare = (rng.random(10000) < .01).astype(int)
NUMS["rare_1pct"] = dict(acc=accuracy_score(y_rare, np.zeros(10000)), bal=balanced_accuracy_score(y_rare, np.zeros(10000)))
print("1%-os célváltozó, konstans 'negatív':", {k: round(v, 3) for k, v in NUMS["rare_1pct"].items()})

'senki nem mond le': {'acc': 0.735, 'bal': 0.5, 'recall': 0.0}
1%-os célváltozó, konstans 'negatív': {'acc': 0.991, 'bal': 0.5}


In [7]:
lr = make_pipeline(make_prep(scale=True), LogisticRegression(max_iter=2000))
p_lr = cross_val_predict(lr, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
tn, fp, fn, tp = confusion_matrix(y_train, p_lr > .5).ravel()
NUMS["lr_confusion"] = dict(TP=int(tp), FP=int(fp), FN=int(fn), TN=int(tn))
NUMS["lr_at_05"] = dict(precision=tp/(tp+fp), recall=tp/(tp+fn), spec=tn/(tn+fp), acc=accuracy_score(y_train, p_lr > .5), bal=balanced_accuracy_score(y_train, p_lr > .5))
print(NUMS["lr_confusion"], {k: round(v, 3) for k, v in NUMS["lr_at_05"].items()})
def thr_table(y, p, ts=(.5, .4, .3, .2)):
    return pd.DataFrame([{"küszöb": t, "megjelölt %": (p > t).mean() * 100, "precision": precision_score(y, p > t), "recall": recall_score(y, p > t),
                          "kiegyens. pontosság": balanced_accuracy_score(y, p > t), "pontosság": accuracy_score(y, p > t)} for t in ts])
thr_lr = thr_table(y_train, p_lr); display(thr_lr.round(3))
o = np.argsort(-p_lr); NUMS["precision_at_k"] = {k: float(y_train.values[o[:k]].mean()) for k in (100, 300, 500, 1000)}
NUMS["lr_roc_auc"], NUMS["lr_pr_auc"] = float(roc_auc_score(y_train, p_lr)), float(average_precision_score(y_train, p_lr)); print(NUMS["precision_at_k"], NUMS["lr_roc_auc"], NUMS["lr_pr_auc"])

{'TP': 822, 'FP': 431, 'FN': 673, 'TN': 3708} {'precision': np.float64(0.656), 'recall': np.float64(0.55), 'spec': np.float64(0.896), 'acc': 0.804, 'bal': 0.723}


,küszöb,megjelölt %,precision,recall,kiegyens. pontosság,pontosság
0,0.5,22.240,0.656,0.550,0.723,0.804
1,0.4,29.553,0.593,0.660,0.748,0.789
2,0.3,37.806,0.542,0.772,0.768,0.766
3,0.2,47.994,0.474,0.858,0.757,0.710


{100: 0.88, 300: 0.7866666666666666, 500: 0.766, 1000: 0.699} 0.8454587693051091 0.6579493887706712


In [8]:
fig, (a, b) = plt.subplots(1, 2, figsize=(13, 5))
fpr_, tpr_, _ = roc_curve(y_train, p_lr); a.plot(fpr_, tpr_, color=BLUE, lw=2.5); a.plot([0, 1], [0, 1], color=INK2, ls="--", lw=1.2)
a.set_title(f"ROC-görbe, AUC = {hu(NUMS['lr_roc_auc'])}", loc="left"); a.set_xlabel("álpozitív arány (FPR)"); a.set_ylabel("valódi pozitív arány (recall)")
pp_, rr_, _ = precision_recall_curve(y_train, p_lr); b.plot(rr_, pp_, color=BLUE, lw=2.5); b.axhline(y_train.mean(), color=INK2, ls="--", lw=1.2)
b.set_title(f"Precision-recall görbe, AP = {hu(NUMS['lr_pr_auc'])}", loc="left"); b.set_ylim(0, 1.02); b.set_xlabel("recall"); b.set_ylabel("precision")
b.text(.52, y_train.mean() + .03, f"alapvonal = {hu(y_train.mean())}", color=INK2, fontsize=14)
for ax_ in (a, b): hu_axes(ax_)
plt.tight_layout(w_pad=4); keep(fig, "metrics_roc_pr")

## 4. A modell-létra

In [9]:
models = {
    "dummy":           make_pipeline(make_prep(), DummyClassifier(strategy="prior")),
    "logreg":          make_pipeline(make_prep(scale=True), LogisticRegression(max_iter=2000)),
    "fa (korlát nélkül)": make_pipeline(make_prep(), DecisionTreeClassifier(random_state=0)),
    "fa (mélység 4)":  make_pipeline(make_prep(), DecisionTreeClassifier(max_depth=4, random_state=0)),
    "random forest":   make_pipeline(make_prep(), RandomForestClassifier(300, random_state=0, n_jobs=-1)),
    "xgboost":         make_pipeline(make_prep(), XGBClassifier(n_estimators=300, learning_rate=.1, max_depth=4, eval_metric="logloss", random_state=0)),
}
KEY = {"logreg": "logreg", "xgboost": "xgboost", "random forest": "random forest"}
scoring = {"PR-AUC": "average_precision", "ROC-AUC": "roc_auc", "bal_acc": "balanced_accuracy", "acc": "accuracy"}
lad, fold_pr = [], {}
for name, m in models.items():
    r = cross_validate(m, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1); fold_pr[name] = r["test_PR-AUC"]
    lad.append({"modell": name, **{k: float(r["test_" + k].mean()) for k in scoring}, **{k + "_sd": float(r["test_" + k].std()) for k in scoring}})
lad = pd.DataFrame(lad); display(lad.round(3)); NUMS["ladder"] = lad.round(4).to_dict("records")

,modell,PR-AUC,ROC-AUC,bal_acc,acc,PR-AUC_sd,ROC-AUC_sd,bal_acc_sd,acc_sd
0,dummy,0.265,0.500,0.500,0.735,0.000,0.000,0.000,0.000
1,logreg,0.661,0.846,0.723,0.804,0.031,0.012,0.015,0.007
2,fa (korlát nélkül),0.380,0.658,0.658,0.729,0.021,0.022,0.022,0.013
3,fa (mélység 4),0.598,0.828,0.690,0.789,0.025,0.014,0.024,0.013
4,random forest,0.609,0.818,0.685,0.782,0.033,0.018,0.023,0.013
5,xgboost,0.653,0.835,0.702,0.791,0.035,0.015,0.021,0.010


In [10]:
fig, ax = plt.subplots(figsize=(9.5, 4.8))
for i, (n_, s_) in enumerate(fold_pr.items()):
    ax.scatter(s_, [i] * len(s_), color=BLUE, s=70, alpha=.9, edgecolor=SURFACE); ax.hlines(i, s_.min(), s_.max(), color=BLUE, lw=1.5, alpha=.5)
ax.set_yticks(range(len(fold_pr))); ax.set_yticklabels(fold_pr); ax.invert_yaxis(); ax.set_xlabel("PR-AUC az 5 CV-fold mindegyikén"); hu_axes(ax, y=False); keep(fig, "ladder_folds")
lr_fit = lr.fit(X_train, y_train)
coef = pd.Series(lr_fit[-1].coef_[0], index=lr_fit[0].get_feature_names_out()).sort_values()
c_ = pd.concat([coef.head(6), coef.tail(6)]); c_.index = [i.split("__", 1)[-1] for i in c_.index]
fig, ax = plt.subplots(figsize=(9.5, 5)); ax.barh(c_.index, c_.values, color=[BLUE if v < 0 else ORANGE for v in c_.values], height=.6)
ax.axvline(0, color=INK2, lw=1); ax.set_xlabel("együttható (a churn log-oddsa)"); ax.grid(False); hu_axes(ax, y=False); keep(fig, "lr_egyutthatok")
NUMS["lr_top_up"] = coef.tail(6).round(2).rename(lambda s: s.split("__", 1)[-1]).to_dict(); NUMS["lr_top_down"] = coef.head(6).round(2).rename(lambda s: s.split("__", 1)[-1]).to_dict()
tree = make_pipeline(make_prep(), DecisionTreeClassifier(max_depth=3, random_state=0)).fit(X_train, y_train)
fig = plt.figure(figsize=(26, 8.5))
ann = plot_tree(tree[-1], feature_names=[n_.split("__", 1)[-1] for n_ in tree[0].get_feature_names_out()], class_names=["marad", "lemond"],
                filled=True, rounded=True, fontsize=13, impurity=False, proportion=True)
for a_ in ann: a_.set_color("#0d2a20")                       # sötét szöveg a világos dobozokban
keep(fig, "tree_depth3")

## 5. Visszacsatolási kör: hibák elemzése

In [11]:
loop_model = models["xgboost"]
p_oof = cross_val_predict(loop_model, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
def metrics(y, p, t=.5):
    return {"accuracy": accuracy_score(y, p > t), "balanced acc": balanced_accuracy_score(y, p > t),
            "recall": recall_score(y, p > t), "precision": precision_score(y, p > t), "PR-AUC": average_precision_score(y, p)}
log = {"alap xgboost @0,5": metrics(y_train, p_oof)}
def error_slices(X, y, p, t=.5, cols=("Contract", "InternetService", "PaymentMethod")):
    d = X.assign(y=y.values, fn=((y.values == 1) & (p <= t)), fp=((y.values == 0) & (p > t)))
    d["tenure_bin"] = pd.cut(d.tenure, [-1, 3, 12, 24, 48, 80]); out = {}
    for c in (*cols, "tenure_bin"):
        g = d.groupby(c, observed=True)
        out[c] = pd.DataFrame({"n": g.size(), "churn arány": g.y.mean(), "kihagyott lemondók / lemondók": g.fn.sum() / g.y.sum(),
                               "álriasztás / maradók": g.fp.sum() / (g.size() - g.y.sum())})
    return out
es = error_slices(X_train, y_train, p_oof)
for c, t in es.items(): display(t.round(2))
NUMS["error_slices"] = {c: t.round(3).reset_index().astype({c: str}).to_dict("records") for c, t in es.items()}
d_ = X_train.assign(churned=y_train.values, p=p_oof.round(2))
display(d_[d_.churned == 0].nlargest(5, "p")[["Contract", "InternetService", "PaymentMethod", "tenure", "MonthlyCharges", "p"]])
display(d_[d_.churned == 1].nsmallest(5, "p")[["Contract", "InternetService", "PaymentMethod", "tenure", "MonthlyCharges", "p"]])

,n,churn arány,kihagyott lemondók / lemondók,álriasztás / maradók
Contract,,,,
Month-to-month,3102,0.43,0.43,0.24
One year,1173,0.11,0.96,0.01
Two year,1359,0.03,1.00,0.00


,n,churn arány,kihagyott lemondók / lemondók,álriasztás / maradók
InternetService,,,,
DSL,1937,0.19,0.72,0.07
Fiber optic,2483,0.42,0.37,0.23
No,1214,0.07,0.86,0.01


,n,churn arány,kihagyott lemondók / lemondók,álriasztás / maradók
PaymentMethod,,,,
Bank transfer (automatic),1244,0.16,0.68,0.04
Credit card (automatic),1213,0.15,0.78,0.04
Electronic check,1891,0.46,0.34,0.29
Mailed check,1286,0.19,0.62,0.06


,n,churn arány,kihagyott lemondók / lemondók,álriasztás / maradók
tenure_bin,,,,
"(-1, 3]",832,0.55,0.26,0.42
"(3, 12]",905,0.40,0.36,0.22
"(12, 24]",818,0.29,0.55,0.14
"(24, 48]",1290,0.20,0.71,0.06
"(48, 80]",1789,0.10,0.91,0.02


,Contract,InternetService,PaymentMethod,tenure,MonthlyCharges,p
5474,Month-to-month,Fiber optic,Electronic check,1,84.60,0.95
3159,Month-to-month,Fiber optic,Electronic check,3,94.85,0.93
5354,Month-to-month,Fiber optic,Electronic check,1,89.35,0.91
459,Month-to-month,Fiber optic,Electronic check,1,75.10,0.91
6963,Month-to-month,Fiber optic,Credit card (automatic),9,102.60,0.91


,Contract,InternetService,PaymentMethod,tenure,MonthlyCharges,p
6724,Two year,DSL,Credit card (automatic),64,70.20,0.0
5947,Two year,No,Mailed check,33,24.50,0.0
5579,Two year,Fiber optic,Bank transfer (automatic),70,93.90,0.0
268,Two year,No,Mailed check,59,19.35,0.0
4819,Two year,No,Bank transfer (automatic),53,19.75,0.0


In [12]:
ts = np.arange(.2, .9, .01)
t_acc = max(ts, key=lambda t: ((p_oof > t) == y_train).mean())                       # 1. HIBA: pontosságra optimalizálunk
log["1. HIBA: küszöb a pontosságra"] = metrics(y_train, p_oof, t_acc)
Xo, yo = RandomOverSampler(random_state=0).fit_resample(X_train, y_train)            # 2. HIBA: előbb felülmintavételezünk ...
p_leaky = cross_val_predict(loop_model, Xo, yo, cv=cv, method="predict_proba")[:, 1]   # ... és csak utána keresztvalidálunk
leaky = metrics(yo, p_leaky)
key = pd.util.hash_pandas_object(Xo, index=False).values
twin = {nm: float(np.mean([np.isin(key[v][yo.values[v] == lab], key[t]).mean() for t, v in cv.split(Xo, yo)])) for lab, nm in [(1, "lemondók"), (0, "maradók")]}
NUMS["leak"] = dict(rows=int(len(Xo)), pos=float(yo.mean()), twin=twin, leaky=leaky, t_acc=float(t_acc))
fixed = imb_pipeline(make_prep(), RandomOverSampler(random_state=0), XGBClassifier(n_estimators=300, learning_rate=.1, max_depth=4, eval_metric="logloss", random_state=0))
p_fix = cross_val_predict(fixed, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
log["felülmintavételezés a foldon belül"] = metrics(y_train, p_fix)
print("accuracy-optimális küszöb", round(t_acc, 2), "| ikerpéldány arány:", {k: round(v, 3) for k, v in twin.items()}); display(pd.DataFrame({**log, "szivárgó (félrevezető)": leaky}).T.round(3))

accuracy-optimális küszöb 0.56 | ikerpéldány arány: {'lemondók': 0.876, 'maradók': 0.004}


,accuracy,balanced acc,recall,precision,PR-AUC
"alap xgboost @0,5",0.791,0.702,0.513,0.629,0.651
1. HIBA: küszöb a pontosságra,0.793,0.683,0.448,0.663,0.651
felülmintavételezés a foldon belül,0.763,0.751,0.724,0.540,0.650
szivárgó (félrevezető),0.822,0.822,0.877,0.789,0.873


In [13]:
addons = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
feats = {
    "+ n_addons":  lambda d: d.assign(n_addons=(d[addons] == "Yes").sum(axis=1)),
    "+ price_gap": lambda d: d.assign(price_gap=np.where(d.tenure > 0, d.TotalCharges / d.tenure.clip(lower=1) - d.MonthlyCharges, 0.0)),
    "+ m2m_fiber": lambda d: d.assign(m2m_fiber=((d.Contract == "Month-to-month") & (d.InternetService == "Fiber optic")).astype(int)),
}
rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=1)
pr = lambda Xf: cross_val_score(loop_model, Xf, y_train, cv=rcv, scoring="average_precision", n_jobs=-1)
base_s = pr(X_train); feat_rows = [{"változat": "alap", "PR-AUC": base_s.mean(), "páros különbség": np.nan, "sd": np.nan}]
for n, fn_ in feats.items():
    s = pr(fn_(X_train)); dlt = s - base_s; feat_rows.append({"változat": n, "PR-AUC": s.mean(), "páros különbség": dlt.mean(), "sd": dlt.std()})
feat = pd.DataFrame(feat_rows); display(feat.round(4)); NUMS["features"] = feat.round(4).to_dict("records")
t_star = max(ts, key=lambda t: balanced_accuracy_score(y_train, p_oof > t))          # a VALÓDI javítás: a működési pont
log["JAVÍTÁS: küszöb a kiegyensúlyozott pontosságra"] = metrics(y_train, p_oof, t_star)
thr_xgb = thr_table(y_train, p_oof, ts=(.5, .4, t_star, .2)); display(pd.DataFrame(log).T.round(3)); display(thr_xgb.round(3))
NUMS["loop"] = {k: {m: float(v) for m, v in d.items()} for k, d in log.items()}; NUMS["t_star"] = float(t_star)
NUMS["thr_xgb"] = thr_xgb.round(4).to_dict("records"); NUMS["thr_lr"] = thr_lr.round(4).to_dict("records")

,változat,PR-AUC,páros különbség,sd
0,alap,0.6469,NaN,NaN
1,+ n_addons,0.6463,-0.0007,0.0069
2,+ price_gap,0.6377,-0.0092,0.0090
3,+ m2m_fiber,0.6472,0.0003,0.0054


,accuracy,balanced acc,recall,precision,PR-AUC
"alap xgboost @0,5",0.791,0.702,0.513,0.629,0.651
1. HIBA: küszöb a pontosságra,0.793,0.683,0.448,0.663,0.651
felülmintavételezés a foldon belül,0.763,0.751,0.724,0.540,0.650
JAVÍTÁS: küszöb a kiegyensúlyozott pontosságra,0.761,0.754,0.739,0.536,0.651


,küszöb,megjelölt %,precision,recall,kiegyens. pontosság,pontosság
0,0.5,21.636,0.629,0.513,0.702,0.791
1,0.4,28.612,0.585,0.631,0.735,0.783
2,0.3,36.599,0.536,0.739,0.754,0.761
3,0.2,45.900,0.477,0.825,0.749,0.714


## 6. Hiperparaméter-hangolás

In [14]:
xgb_space = {"xgbclassifier__n_estimators": randint(50, 400), "xgbclassifier__learning_rate": loguniform(.01, .3),
             "xgbclassifier__max_depth": randint(2, 6), "xgbclassifier__min_child_weight": randint(1, 30),
             "xgbclassifier__subsample": uniform(.6, .4), "xgbclassifier__colsample_bytree": uniform(.4, .6),
             "xgbclassifier__reg_lambda": loguniform(.1, 20)}
rf_space = {"randomforestclassifier__max_depth": randint(3, 12), "randomforestclassifier__min_samples_leaf": randint(5, 60),
            "randomforestclassifier__max_features": uniform(.1, .6)}
searches = {}
for name, space, n_iter in [("xgboost", xgb_space, 30), ("random forest", rf_space, 25)]:
    searches[name] = RandomizedSearchCV(models[name], space, n_iter=n_iter, scoring="average_precision", cv=cv, random_state=0, n_jobs=-1).fit(X_train, y_train)
    print(name, "legjobb CV PR-AUC", round(searches[name].best_score_, 4), {k.split("__")[1]: (round(float(v), 3)) for k, v in searches[name].best_params_.items()})
fresh = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=7)              # foldok, amiket a keresés sosem látott
tune = {}
for name, tuned in [("logreg", None), ("random forest", searches["random forest"]), ("xgboost", searches["xgboost"])]:
    s0 = cross_val_score(models[name], X_train, y_train, cv=fresh, scoring="average_precision", n_jobs=-1)
    s1 = s0 if tuned is None else cross_val_score(tuned.best_estimator_, X_train, y_train, cv=fresh, scoring="average_precision", n_jobs=-1)
    tune[name] = dict(default=float(s0.mean()), default_sd=float(s0.std()), tuned=float(s1.mean()), tuned_sd=float(s1.std()), search_best=None if tuned is None else float(tuned.best_score_))
display(pd.DataFrame(tune).T.round(4)); NUMS["tune"] = tune
NUMS["best_params_xgb"] = {k.split("__")[1]: float(v) for k, v in searches["xgboost"].best_params_.items()}
fig, ax = plt.subplots(figsize=(10, 4.6)); ys = np.arange(len(tune))
ax.errorbar([v["default"] for v in tune.values()], ys + .13, xerr=[v["default_sd"] for v in tune.values()], fmt="o", color=INK2, ecolor=INK2, ms=11, capsize=5, label="alapértelmezett")
ax.errorbar([v["tuned"] for v in tune.values()], ys - .13, xerr=[v["tuned_sd"] for v in tune.values()], fmt="o", color=BLUE, ecolor=BLUE, ms=11, capsize=5, label="hangolt")
ax.set_yticks(ys); ax.set_yticklabels(tune); ax.invert_yaxis(); ax.set_xlabel("PR-AUC új CV-foldokon (átlag ± szórás)"); ax.legend(frameon=False, loc="lower left"); hu_axes(ax, y=False); keep(fig, "tuning_elotte_utana")

xgboost legjobb CV PR-AUC 0.6711 {'colsample_bytree': 0.688, 'learning_rate': 0.038, 'max_depth': 3.0, 'min_child_weight': 14.0, 'n_estimators': 122.0, 'reg_lambda': 0.159, 'subsample': 0.608}


/home/karpatika92/nasz_data_science_course/.claude/worktrees/lesson04-klasszifikacio/.venv/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/karpatika92/nasz_data_science_course/.claude/worktrees/lesson04-klasszifikacio/.venv/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/karpatika92/nasz_data_science_course/.claude/worktrees/lesson04-klasszifikacio/.venv/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Paral

/home/karpatika92/nasz_data_science_course/.claude/worktrees/lesson04-klasszifikacio/.venv/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/karpatika92/nasz_data_science_course/.claude/worktrees/lesson04-klasszifikacio/.venv/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/karpatika92/nasz_data_science_course/.claude/worktrees/lesson04-klasszifikacio/.venv/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Paral

random forest legjobb CV PR-AUC 0.6681 {'max_depth': 7.0, 'max_features': 0.384, 'min_samples_leaf': 16.0}


,default,default_sd,tuned,tuned_sd,search_best
logreg,0.6614,0.0276,0.6614,0.0276,NaN
random forest,0.6193,0.0284,0.6681,0.0248,0.6681
xgboost,0.6520,0.0276,0.6717,0.0253,0.6711


## 7. Záró kiértékelés (a teszthalmaz, egyszer)

In [15]:
cands = {"logreg": models["logreg"], "xgboost (hangolt)": searches["xgboost"].best_estimator_}
cv_scores = {n: float(cross_val_score(m, X_train, y_train, cv=fresh, scoring="average_precision", n_jobs=-1).mean()) for n, m in cands.items()}
final_name = max(cv_scores, key=cv_scores.get); print("CV PR-AUC:", {k: round(v, 4) for k, v in cv_scores.items()}, "-> végső:", final_name)
p_oof_final = cross_val_predict(cands[final_name], X_train, y_train, cv=cv, method="predict_proba")[:, 1]
t_final = max(ts, key=lambda t: balanced_accuracy_score(y_train, p_oof_final > t)); print("küszöb a train OOF-ból:", round(t_final, 2))
test_p = {n: m.fit(X_train, y_train).predict_proba(X_test)[:, 1] for n, m in cands.items()}; p_te = test_p[final_name]
res = pd.DataFrame({"teszt @0,5": metrics(y_test, p_te, .5), f"teszt @{hu(t_final)}": metrics(y_test, p_te, t_final)}).T; display(res.round(3))
tn, fp, fn, tp = confusion_matrix(y_test, p_te > t_final).ravel()
rng = np.random.default_rng(0); yt = y_test.values; n = len(yt); idx = [rng.integers(0, n, n) for _ in range(1000)]
ap = lambda p, i: average_precision_score(yt[i], p[i])
ci = np.percentile([ap(p_te, i) for i in idx], [2.5, 97.5]); other = [k for k in cands if k != final_name][0]
diff = np.array([ap(p_te, i) - ap(test_p[other], i) for i in idx])
NUMS["final"] = dict(final_name=final_name, t_final=float(t_final), cv_scores=cv_scores, test_n=int(n), test_pos=int(yt.sum()),
    test_at_05=metrics(y_test, p_te, .5), test_at_t=metrics(y_test, p_te, t_final), confusion=dict(TP=int(tp), FP=int(fp), FN=int(fn), TN=int(tn)),
    test_pr_auc={k: float(average_precision_score(y_test, v)) for k, v in test_p.items()}, boot_ci=[float(v) for v in ci],
    paired=dict(vs=other, mean=float(diff.mean()), lo=float(np.percentile(diff, 2.5)), hi=float(np.percentile(diff, 97.5))))
print(json.dumps(NUMS["final"], indent=1))
fig, ax = plt.subplots(figsize=(6.8, 5)); pp_, rr_, _ = precision_recall_curve(y_test, p_te); ax.plot(rr_, pp_, color=BLUE, lw=2.5); ax.axhline(y_test.mean(), color=INK2, ls="--", lw=1.2)
ax.scatter([recall_score(y_test, p_te > t_final)], [precision_score(y_test, p_te > t_final)], color=ORANGE, s=150, zorder=3, label=f"választott küszöb: {hu(t_final)}")
ax.set_ylim(0, 1.02); ax.set_xlabel("recall"); ax.set_ylabel("precision"); ax.legend(frameon=False, loc="upper right"); hu_axes(ax); keep(fig, "final_pr_gorbe")

CV PR-AUC: {'logreg': 0.6614, 'xgboost (hangolt)': 0.6717} -> végső: xgboost (hangolt)


küszöb a train OOF-ból: 0.23


,accuracy,balanced acc,recall,precision,PR-AUC
"teszt @0,5",0.806,0.712,0.511,0.680,0.663
"teszt @0,23",0.730,0.764,0.837,0.494,0.663


{
 "final_name": "xgboost (hangolt)",
 "t_final": 0.23000000000000004,
 "cv_scores": {
  "logreg": 0.661377607667223,
  "xgboost (hangolt)": 0.6717174865629637
 },
 "test_n": 1409,
 "test_pos": 374,
 "test_at_05": {
  "accuracy": 0.8062455642299503,
  "balanced acc": 0.7118693327133225,
  "recall": 0.5106951871657754,
  "precision": 0.6797153024911032,
  "PR-AUC": 0.6631405423534303
 },
 "test_at_t": {
  "accuracy": 0.7295954577714692,
  "balanced acc": 0.7638598258802862,
  "recall": 0.8368983957219251,
  "precision": 0.49447077409162715,
  "PR-AUC": 0.6631405423534303
 },
 "confusion": {
  "TP": 313,
  "FP": 320,
  "FN": 61,
  "TN": 715
 },
 "test_pr_auc": {
  "logreg": 0.6337024099769349,
  "xgboost (hangolt)": 0.6631405423534303
 },
 "boot_ci": [
  0.6153692017168684,
  0.7141035870022372
 ],
 "paired": {
  "vs": "logreg",
  "mean": 0.02921646463748489,
  "lo": 0.010245404068410027,
  "hi": 0.048179790283892764
 }
}


## Ábrák és táblázatok a slide-okhoz

In [16]:
def table_png(df, name, widths=None, hl=None, fs=17):
    """Sötét témájú táblázat-ábra (a deck-motornak nincs táblázat-blokkja)."""
    widths = [max(len(str(v)) for v in [c, *df[c]]) * 0.155 + 0.8 for c in df.columns]      # oszlopszélesség a szöveghosszból
    fig, ax = plt.subplots(figsize=(sum(widths), .62 * (len(df) + 1))); ax.axis("off")
    t = ax.table(cellText=df.values.tolist(), colLabels=list(df.columns), loc="center", cellLoc="center", colWidths=[w / sum(widths) for w in widths] if widths else None)
    t.auto_set_font_size(False); t.set_fontsize(fs); t.scale(1, 1.9)
    for (r, c), cell in t.get_celld().items():
        cell.set_edgecolor(GRID); cell.set_linewidth(1)
        if r == 0: cell.set_facecolor("#123626"); cell.get_text().set_color("#d1a13a"); cell.get_text().set_fontweight("bold")
        else:
            cell.set_facecolor("#16442f" if hl and (r - 1) in hl else SURFACE); cell.get_text().set_color(INK)
        if c == 0: cell.set_text_props(ha="left"); cell._loc = "left"
    keep(fig, name)
f3 = lambda v: hu(v, 3)
t = lad.copy(); table_png(pd.DataFrame({"modell": t.modell, "PR-AUC": [f"{f3(a)} ± {f3(b)}" for a, b in zip(t["PR-AUC"], t["PR-AUC_sd"])],
    "ROC-AUC": [f"{f3(a)} ± {f3(b)}" for a, b in zip(t["ROC-AUC"], t["ROC-AUC_sd"])], "kiegyens. pont. @0,5": [f"{f3(a)} ± {f3(b)}" for a, b in zip(t.bal_acc, t.bal_acc_sd)],
    "pontosság @0,5": [f"{f3(a)} ± {f3(b)}" for a, b in zip(t.acc, t.acc_sd)]}), "tbl_letra", widths=[3.0, 2.6, 2.6, 2.9, 2.6])
table_png(pd.DataFrame({"küszöb": [hu(v, 2) for v in thr_lr["küszöb"]], "megjelölt": [f"{hu(v, 1)}%" for v in thr_lr["megjelölt %"]], "precision": [f3(v) for v in thr_lr.precision],
    "recall": [f3(v) for v in thr_lr.recall], "kiegyens. pont.": [f3(v) for v in thr_lr["kiegyens. pontosság"]], "pontosság": [f3(v) for v in thr_lr["pontosság"]]}), "tbl_kuszob_lr", widths=[1.6, 1.9, 1.9, 1.9, 2.5, 2.1])
lg = pd.DataFrame(log).T
def loop_tbl(rows, name, hl=None):
    d = lg.loc[rows]; table_png(pd.DataFrame({"": rows, "pontosság": [f3(v) for v in d.accuracy], "kiegyens. pont.": [f3(v) for v in d["balanced acc"]], "recall": [f3(v) for v in d.recall],
        "precision": [f3(v) for v in d.precision], "PR-AUC": [f3(v) for v in d["PR-AUC"]]}), name, widths=[5.6, 1.8, 2.4, 1.6, 1.8, 1.8], hl=hl)
loop_tbl(["alap xgboost @0,5", "1. HIBA: küszöb a pontosságra"], "tbl_hiba1")
lk = pd.DataFrame({"alap xgboost @0,5": lg.loc["alap xgboost @0,5"], "szivárgó: előbb mintavétel, aztán CV": pd.Series(leaky), "mintavétel a foldon belül": lg.loc["felülmintavételezés a foldon belül"]}).T
table_png(pd.DataFrame({"": lk.index, "pontosság": [f3(v) for v in lk.accuracy], "kiegyens. pont.": [f3(v) for v in lk["balanced acc"]], "recall": [f3(v) for v in lk.recall],
    "precision": [f3(v) for v in lk.precision], "PR-AUC": [f3(v) for v in lk["PR-AUC"]]}), "tbl_hiba2", widths=[6.2, 1.8, 2.4, 1.6, 1.8, 1.8], hl=[1])
loop_tbl(list(lg.index), "tbl_utazas", hl=[3])
table_png(pd.DataFrame({"változat": feat["változat"], "PR-AUC (CV)": [f3(v) for v in feat["PR-AUC"]],
    "páros különbség az alaphoz": ["-"] + [f"{'+' if a >= 0 else '−'}{hu(abs(a), 4)} ± {hu(b, 4)}" for a, b in zip(feat["páros különbség"][1:], feat.sd[1:])]}), "tbl_jellemzok", widths=[2.8, 2.6, 4.4])
fin = pd.DataFrame(NUMS["final"]).T if False else None
rr = res.copy(); table_png(pd.DataFrame({"": rr.index, "pontosság": [f3(v) for v in rr.accuracy], "kiegyens. pont.": [f3(v) for v in rr["balanced acc"]], "recall": [f3(v) for v in rr.recall],
    "precision": [f3(v) for v in rr.precision], "PR-AUC": [f3(v) for v in rr["PR-AUC"]]}), "tbl_vegso", widths=[3.0, 1.8, 2.4, 1.6, 1.8, 1.8], hl=[1])
fig, ax = plt.subplots(figsize=(10.5, 4.8))
for col, colr in [("accuracy", INK2), ("balanced acc", BLUE), ("recall", ORANGE)]:
    ax.plot(ts, [metrics(y_train, p_oof, t)[col] for t in ts], color=colr, lw=2.8, label={"accuracy": "pontosság", "balanced acc": "kiegyensúlyozott pontosság", "recall": "recall"}[col])
ax.axvline(t_acc, color=INK2, ls="--", lw=1.2); ax.axvline(t_star, color=BLUE, ls="--", lw=1.2)
ax.text(t_acc + .01, .98, f"a pontosság csúcsa\n{hu(t_acc)}-nál", color=INK2, fontsize=14, va="top"); ax.text(t_star + .01, .98, f"kiegyens. pont.\ncsúcsa {hu(t_star)}-nál", color=BLUE, fontsize=14, va="top")
ax.set_xlabel("döntési küszöb"); ax.set_ylim(.15, 1.0); ax.legend(frameon=False, loc="lower left"); hu_axes(ax); keep(fig, "loop_kuszob_gorbek")
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8), gridspec_kw={"width_ratios": [1, 1.4]})
lab = {"Month-to-month": "havi", "One year": "1 éves", "Two year": "2 éves", "(-1, 3]": "0-3", "(3, 12]": "4-12", "(12, 24]": "13-24", "(24, 48]": "25-48", "(48, 80]": "49+"}
for ax, k, title in [(axes[0], "Contract", "Szerződés szerint"), (axes[1], "tenure_bin", "Tenure szerint (hónap)")]:
    t = es[k]; x = np.arange(len(t)); w = .38
    ax.bar(x - w/2, t["kihagyott lemondók / lemondók"], w, color=ORANGE, label="kihagyott lemondók / lemondók")
    ax.bar(x + w/2, t["álriasztás / maradók"], w, color=BLUE, label="álriasztás / maradók")
    ax.set_xticks(x); ax.set_xticklabels([lab.get(str(i), str(i)) for i in t.index]); ax.set_ylim(0, 1.08); ax.set_title(title, loc="left"); ax.grid(axis="x", visible=False); hu_axes(ax, x=False)
h_, l_ = axes[0].get_legend_handles_labels(); fig.legend(h_, l_, loc="lower center", ncol=2, frameon=False); fig.subplots_adjust(wspace=.14, bottom=.2); keep(fig, "loop_hibak")
vals = [("tisztességes alap", log["alap xgboost @0,5"]["PR-AUC"], BLUE), ("előbb mintavétel, aztán CV", leaky["PR-AUC"], ORANGE), ("mintavétel a foldon belül", log["felülmintavételezés a foldon belül"]["PR-AUC"], BLUE)]
fig, ax = plt.subplots(figsize=(11, 4.6)); ax.bar([v[0] for v in vals], [v[1] for v in vals], color=[v[2] for v in vals], width=.55)
for i, v in enumerate(vals): ax.text(i, v[1] + .015, hu(v[1]), ha="center", fontsize=19, color=INK)
ax.set_ylim(0, 1); ax.set_ylabel("keresztvalidált PR-AUC"); ax.grid(axis="x", visible=False); hu_axes(ax, x=False); keep(fig, "loop_szivargas")
Path("numbers.json").write_text(json.dumps(NUMS, indent=1, default=float)); print("kész:", sorted(p.name for p in OUT.glob("*.png")))

kész: ['eda_internet_fizetes.png', 'eda_szerzodes.png', 'eda_tenure.png', 'final_pr_gorbe.png', 'ladder_folds.png', 'loop_hibak.png', 'loop_kuszob_gorbek.png', 'loop_szivargas.png', 'lr_egyutthatok.png', 'metrics_roc_pr.png', 'tbl_hiba1.png', 'tbl_hiba2.png', 'tbl_jellemzok.png', 'tbl_kuszob_lr.png', 'tbl_letra.png', 'tbl_utazas.png', 'tbl_vegso.png', 'tree_depth3.png', 'tuning_elotte_utana.png']
